In [1]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

# === 数据准备（复用上节） ===
np.random.seed(42)
data = np.sin(np.linspace(0, 8 * np.pi, 500)) + 0.1 * np.random.randn(500)


def create_sliding_window(data, window_size=20):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size])
        y.append(data[i + window_size])
    return np.array(X), np.array(y)


X, y = create_sliding_window(data)
split = int(len(X) * 0.8)

X_train = torch.FloatTensor(X[:split]).unsqueeze(-1)
X_test = torch.FloatTensor(X[split:]).unsqueeze(-1)
y_train = torch.FloatTensor(y[:split])
y_test = torch.FloatTensor(y[split:])

In [2]:
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

In [3]:
class LSTMPredictor(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1]).squeeze(-1)

In [4]:
model = LSTMPredictor()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

In [5]:
for epoch in range(50):

    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        pred = model(xb)
        loss = criterion(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1:2d} | MSE Loss: {avg_loss:.6f}")

Epoch 10 | MSE Loss: 0.015062
Epoch 20 | MSE Loss: 0.014452
Epoch 30 | MSE Loss: 0.013599
Epoch 40 | MSE Loss: 0.013521
Epoch 50 | MSE Loss: 0.012364


In [6]:
torch.save(model.state_dict(), "lstm_predictor.pt")
print("模型已保存")

模型已保存
